In [ ]:
# Draggable points -> Python -> redrawn curve, in a JupyterLab cell (HoloViews).
# Requires: pip install holoviews bokeh panel
# Paste into a single notebook cell and run; the plot appears in the cell output.

import numpy as np
import holoviews as hv
from holoviews import streams

hv.extension("bokeh")   # use hv.extension("bokeh", inline=True) to work fully offline

# 1. Draggable points: drag to move, tap empty space to add, select + Backspace to delete
ctrl = hv.Points([(1, 1), (3, 9), (7, 9), (9, 1)]).opts(
    size=10, color="black",
    active_tools=["point_draw"],
    xlim=(0, 10), ylim=(0, 10),
    width=600, height=400,
)

# 2. Stream: pts.data always holds the current {"x": [...], "y": [...]}
pts = streams.PointDraw(source=ctrl, data=ctrl.columns(), num_objects=20)


# 3. Your function: gets the slider value(s) and the stream's data
def curve_from_points(samples, data):
    P = np.column_stack([data.get("x", []), data.get("y", [])])
    if len(P) < 2:
        return hv.Curve([])

    # Bezier curve through the control points (de Casteljau)
    t = np.linspace(0, 1, samples)[:, None, None]
    B = np.repeat(P[None], samples, axis=0)
    while B.shape[1] > 1:
        B = (1 - t) * B[:, :-1] + t * B[:, 1:]

    return hv.Curve((B[:, 0, 0], B[:, 0, 1])).opts(color="steelblue", line_width=2)


# 4. Connect: kdims become sliders, streams re-run the function when points move
samples = hv.Dimension("samples", values=list(range(10, 201, 10)), default=100)
curve = hv.DynamicMap(curve_from_points, kdims=[samples], streams=[pts])

curve * ctrl   # last line of the cell -> displayed in the notebook

# After dragging, in another cell:  pts.data
